<p align="right"><a href="./C1_W3_Lab02_Sigmoid_function_Soln.ipynb">English</a> | <strong>中文</strong></p>

# Optional Lab: 逻辑回归（Logistic Regression）

在这个非评分实验中，你将
- 探索 sigmoid 函数（也叫 logistic 函数）
- 探索逻辑回归——它使用 sigmoid 函数

In [1]:
import numpy as np
%matplotlib widget
import matplotlib.pyplot as plt
from plt_one_addpt_onclick import plt_one_addpt_onclick
from lab_utils_common import draw_vthresh
plt.style.use('./deeplearning.mplstyle')

## Sigmoid（逻辑）函数
<img align="left" src="./images/C1_W3_LogisticRegression_left.png"     style=" width:300px; padding: 10px; " >如课程视频所述，对分类任务，我们可以先用线性回归模型 $f_{\mathbf{w},b}(\mathbf{x}^{(i)}) = \mathbf{w} \cdot  \mathbf{x}^{(i)} + b$ 来根据 $x$ 预测 $y$。
- 但由于输出变量 $y$ 只取 0 或 1，我们希望分类模型的预测落在 0 到 1 之间。
- 这可以通过 "sigmoid 函数" 实现，它把所有输入值映射到 0 到 1 之间。

我们来实现 sigmoid 函数，亲眼看看。

## Sigmoid 函数公式

sigmoid 函数的公式如下——

$g(z) = \frac{1}{1+e^{-z}}\tag{1}$

在逻辑回归中，z（sigmoid 函数的输入）是线性回归模型的输出。
- 单个样本时，$z$ 是标量。
- 多个样本时，$z$ 可以是一个含 $m$ 个值的向量，每个样本一个值。
- sigmoid 函数的实现应当同时支持这两种输入形式。
我们用 Python 来实现它。

NumPy 有一个函数 [`exp()`](https://numpy.org/doc/stable/reference/generated/numpy.exp.html)，能方便地计算输入数组（`z`）中所有元素的指数（$e^{z}$）。

它对单个数字作为输入也适用，如下所示。

In [2]:
# Input is an array. 
input_array = np.array([1,2,3])
exp_array = np.exp(input_array)

print("Input to exp:", input_array)
print("Output of exp:", exp_array)

# Input is a single number
input_val = 1  
exp_val = np.exp(input_val)

print("Input to exp:", input_val)
print("Output of exp:", exp_val)

Input to exp: [1 2 3]
Output of exp: [ 2.72  7.39 20.09]
Input to exp: 1
Output of exp: 2.718281828459045


`sigmoid` 函数用 python 实现如下面的 cell 所示。

In [3]:
def sigmoid(z):
    """
    Compute the sigmoid of z

    Args:
        z (ndarray): A scalar, numpy array of any size.

    Returns:
        g (ndarray): sigmoid(z), with the same shape as z
         
    """

    g = 1/(1+np.exp(-z))
   
    return g

我们看看这个函数对各种 `z` 值的输出。

In [4]:
# Generate an array of evenly spaced values between -10 and 10
z_tmp = np.arange(-10,11)

# Use the function implemented above to get the sigmoid values
y = sigmoid(z_tmp)

# Code for pretty printing the two arrays next to each other
np.set_printoptions(precision=3) 
print("Input (z), Output (sigmoid(z))")
print(np.c_[z_tmp, y])

Input (z), Output (sigmoid(z))
[[-1.000e+01  4.540e-05]
 [-9.000e+00  1.234e-04]
 [-8.000e+00  3.354e-04]
 [-7.000e+00  9.111e-04]
 [-6.000e+00  2.473e-03]
 [-5.000e+00  6.693e-03]
 [-4.000e+00  1.799e-02]
 [-3.000e+00  4.743e-02]
 [-2.000e+00  1.192e-01]
 [-1.000e+00  2.689e-01]
 [ 0.000e+00  5.000e-01]
 [ 1.000e+00  7.311e-01]
 [ 2.000e+00  8.808e-01]
 [ 3.000e+00  9.526e-01]
 [ 4.000e+00  9.820e-01]
 [ 5.000e+00  9.933e-01]
 [ 6.000e+00  9.975e-01]
 [ 7.000e+00  9.991e-01]
 [ 8.000e+00  9.997e-01]
 [ 9.000e+00  9.999e-01]
 [ 1.000e+01  1.000e+00]]


左列是 `z`，右列是 `sigmoid(z)`。可以看到，sigmoid 的输入值范围是 -10 到 10，输出值范围是 0 到 1。

现在，我们用 `matplotlib` 库把这个函数画出来。

In [5]:
# Plot z vs sigmoid(z)
fig,ax = plt.subplots(1,1,figsize=(5,3))
ax.plot(z_tmp, y, c="b")

ax.set_title("Sigmoid function")
ax.set_ylabel('sigmoid(z)')
ax.set_xlabel('z')
draw_vthresh(ax,0)

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

可以看到，当 `z` 取很大的负值时 sigmoid 函数趋近 `0`，当 `z` 取很大的正值时趋近 `1`。

## 逻辑回归
<img align="left" src="./images/C1_W3_LogisticRegression_right.png"     style=" width:300px; padding: 10px; " > 逻辑回归模型把 sigmoid 应用到我们熟悉的线性回归模型上，如下所示：

$$ f_{\mathbf{w},b}(\mathbf{x}^{(i)}) = g(\mathbf{w} \cdot \mathbf{x}^{(i)} + b ) \tag{2} $$

  其中

  $g(z) = \frac{1}{1+e^{-z}}\tag{3}$


我们把逻辑回归应用到肿瘤分类这个类别型数据的例子上。
先加载样本和参数的初值。

In [6]:
x_train = np.array([0., 1, 2, 3, 4, 5])
y_train = np.array([0,  0, 0, 1, 1, 1])

w_in = np.zeros((1))
b_in = 0

试试下面的步骤：
- 点击 'Run Logistic Regression' 为给定训练数据找到最佳逻辑回归模型
    - 注意得到的模型对数据拟合得相当好。
    - 注意橙色线是上面的 '$z$'，即 $\mathbf{w} \cdot \mathbf{x}^{(i)} + b$。它与线性回归模型里的那条线不一样。
再通过应用一个 *阈值（threshold）* 进一步改进结果。
- 勾选 'Toggle 0.5 threshold' 框，查看应用阈值后的预测。
    - 这些预测看起来不错，与数据吻合
    - 现在，在肿瘤很大的范围（接近 10）再加几个数据点，重新运行逻辑回归。
    - 与线性回归模型不同，这个模型仍能做出正确预测

In [9]:
plt.close('all') 
addpt = plt_one_addpt_onclick( x_train,y_train, w_in, b_in, logistic=True)

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

## 恭喜！
你已经探索了 sigmoid 函数在逻辑回归中的使用。